# Partie II — CNN et vision par ordinateur

**Thème :** classification d'images avec réseaux de neurones convolutionnels.

Ce notebook couvre :

- limites du MLP pour les images ;
- calculs dimensionnels de convolution/pooling ;
- implémentation manuelle de corrélation croisée, max-pooling et average-pooling ;
- comparaison avec PyTorch ;
- entraînement d'un CNN inspiré de LeNet ;
- comparaison MLP vs CNN ;
- influence du padding, pooling, nombre de filtres et convolution 1×1 ;
- visualisation de cartes de caractéristiques.

In [ ]:
from pathlib import Path
import os, sys, json

# Exécuter ce notebook depuis le dossier project ou depuis project/notebooks fonctionne.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)
print("Python path ready")

In [ ]:
from types import SimpleNamespace
from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn

from src.common import ensure_dir, get_device, set_seed, save_json
from part2_cnn.train_cnn import (
    output_size,
    xcorr2d_manual,
    max_pool2d_manual,
    avg_pool2d_manual,
    verify_manual_ops,
    load_fashion_mnist,
    load_digits_dataset,
    MLPImageBaseline,
    LeNetVariant,
    train_model,
    visualize_feature_maps,
)

set_seed(42)
device = get_device()
print("Device utilisé:", device)

## 1. Calculs dimensionnels

Pour une convolution 2D, la taille de sortie est :

$$
O = \left\lfloor rac{W - K + 2P}{S} ightfloor + 1
$$

avec :

- $W$ : taille de l'entrée ;
- $K$ : taille du noyau ;
- $P$ : padding ;
- $S$ : stride.

In [ ]:
examples = [
    {"W": 28, "K": 3, "P": 0, "S": 1},
    {"W": 28, "K": 3, "P": 1, "S": 1},
    {"W": 28, "K": 5, "P": 0, "S": 2},
]

for e in examples:
    O = output_size(e["W"], e["K"], e["P"], e["S"])
    print(f"Input={e['W']} | Kernel={e['K']} | Padding={e['P']} | Stride={e['S']} => Output={O}")

## 2. Opérations manuelles : corrélation croisée et pooling

Cette partie montre que les couches CNN sont des opérations mathématiques explicites, pas seulement des boîtes noires.

In [ ]:
X = torch.tensor([
    [1., 2., 3.],
    [4., 5., 6.],
    [7., 8., 9.],
])
K = torch.tensor([
    [1., 0.],
    [0., -1.],
])

print("Entrée X:
", X)
print("Noyau K:
", K)
print("Corrélation croisée manuelle:
", xcorr2d_manual(X, K))
print("Max pooling 2x2:
", max_pool2d_manual(X, (2, 2)))
print("Average pooling 2x2:
", avg_pool2d_manual(X, (2, 2)))

## 3. Dataset image

Par défaut, on utilise **Fashion-MNIST**. Si le téléchargement échoue, le notebook utilise automatiquement le dataset `digits` de scikit-learn comme fallback technique.

In [ ]:
args = SimpleNamespace(
    dataset="fashionmnist",
    epochs=5,
    batch_size=64,
    max_train=12000,
    max_test=2000,
    lr=1e-3,
    weight_decay=1e-4,
)

base_out = ensure_dir(ROOT / "results" / "part2_cnn")
verify_manual_ops(base_out)

try:
    train_loader, val_loader, test_loader, image_size, in_channels, num_classes, class_names = load_fashion_mnist(
        args.batch_size, args.max_train, args.max_test
    )
    dataset_used = "Fashion-MNIST"
except Exception as exc:
    print("Fashion-MNIST indisponible, fallback vers sklearn digits:", exc)
    train_loader, val_loader, test_loader, image_size, in_channels, num_classes, class_names = load_digits_dataset(args.batch_size)
    dataset_used = "sklearn digits"

print("Dataset:", dataset_used)
print("Image size:", image_size)
print("Channels:", in_channels)
print("Classes:", class_names)
print("Train:", len(train_loader.dataset), "Validation:", len(val_loader.dataset), "Test:", len(test_loader.dataset))

## 4. Comparaison expérimentale

On compare un MLP baseline avec plusieurs variantes CNN. L'objectif est d'étudier les effets de l'architecture, pas seulement de maximiser l'accuracy.

In [ ]:
experiments = {
    "mlp_baseline": MLPImageBaseline(image_size * image_size * in_channels, num_classes),
    "cnn_baseline_maxpool": LeNetVariant(image_size, in_channels, num_classes, pooling="max"),
    "cnn_average_pooling": LeNetVariant(image_size, in_channels, num_classes, pooling="avg"),
    "cnn_with_padding": LeNetVariant(image_size, in_channels, num_classes, padding=1, pooling="max"),
    "cnn_fewer_filters": LeNetVariant(image_size, in_channels, num_classes, filters1=16, filters2=32, pooling="max"),
    "cnn_with_1x1_conv": LeNetVariant(image_size, in_channels, num_classes, padding=1, use_conv1x1=True, pooling="max"),
}

rows = []
for name, model in experiments.items():
    print("
" + "=" * 80)
    print(f"Experiment: {name}")
    out_dir = ensure_dir(base_out / name)
    metrics = train_model(model, train_loader, val_loader, test_loader, args, device, out_dir, class_names)
    rows.append({"experiment": name, **metrics})
    if isinstance(model, LeNetVariant) and name == "cnn_baseline_maxpool":
        visualize_feature_maps(model, test_loader, device, out_dir)

summary = pd.DataFrame(rows)
summary.to_csv(base_out / "summary_metrics.csv", index=False)
summary

## 5. Visualisation des courbes et cartes de caractéristiques

Les cartes de caractéristiques montrent les activations produites par les filtres convolutionnels. Elles servent à interpréter ce que le CNN extrait progressivement de l'image.

In [ ]:
from IPython.display import Image, display

selected_experiment = "cnn_baseline_maxpool"
exp_dir = base_out / selected_experiment

display(Image(filename=str(exp_dir / "training_curves.png")))
display(Image(filename=str(exp_dir / "confusion_matrix.png")))
feature_maps_path = exp_dir / "feature_maps_conv1.png"
if feature_maps_path.exists():
    display(Image(filename=str(feature_maps_path)))
print((exp_dir / "classification_report.txt").read_text(encoding="utf-8"))

## 6. Analyse critique à intégrer au rapport

Un CNN est plus pertinent qu'un MLP pour les images parce qu'il exploite la structure spatiale : localité des pixels, partage des poids et hiérarchie des représentations. Le padding préserve mieux les dimensions et les informations de bord, le stride réduit la résolution plus rapidement, le pooling rend les représentations plus robustes aux petites translations, et la profondeur permet d'apprendre des motifs de plus en plus abstraits. Un MLP peut fonctionner sur des images simples, mais il détruit la géométrie de l'image en l'aplatissant.